In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import os
from datetime import datetime
import glob

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("="*80)
print("🚗 ПРЕДСКАЗАНИЕ ЦЕН АВТОМОБИЛЕЙ - ПОЛНЫЙ ПАЙПЛАЙН")
print("="*80)

🚗 ПРЕДСКАЗАНИЕ ЦЕН АВТОМОБИЛЕЙ - ПОЛНЫЙ ПАЙПЛАЙН


In [30]:
print("\n" + "="*80)
print("ШАГ 1: ЗАГРУЗКА ДАННЫХ")
print("="*80)

def load_data_from_folders_glob(base_path, filename="data.csv"):
    """
    Загружает все data.csv из подпапок base_path
    """
    csv_files = glob.glob(os.path.join(base_path, "**", filename), recursive=True)
    print(f"Найдено файлов: {len(csv_files)}")
    
    dataframes = []
    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)
            dataframes.append(df)
            print(f"  Загружен: {os.path.basename(os.path.dirname(file_path))} (строк: {len(df)})")
        except Exception as e:
            print(f"  Ошибка при загрузке {file_path}: {e}")
    
    if dataframes:
        X_train = pd.concat(dataframes, ignore_index=True)
        print(f"\n✅ Итоговый размер X_train: {X_train.shape}")
        return X_train
    else:
        print("❌ Файлы data.csv не найдены!")
        return pd.DataFrame()

# Загружаем данные
base_directory = "./"  # Путь к папке с 48 подпапками
X_train = load_data_from_folders_glob(base_directory)
y_train = pd.read_csv('./y_train_base.csv')
X_test = pd.read_csv('./X_test_base.csv')

print("\n" + "="*80)
print("РАЗМЕРЫ ДАННЫХ ПОСЛЕ ЗАГРУЗКИ")
print("="*80)
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")


ШАГ 1: ЗАГРУЗКА ДАННЫХ
Найдено файлов: 18
  Загружен: dataset_10 (строк: 1069)
  Загружен: dataset_2 (строк: 1026)
  Загружен: dataset_7 (строк: 1079)
  Загружен: dataset_30 (строк: 1108)
  Загружен: dataset_1 (строк: 174)
  Загружен: dataset_40 (строк: 1071)
  Загружен: dataset_29 (строк: 1063)
  Загружен: dataset_21 (строк: 996)
  Загружен: dataset_12 (строк: 1014)
  Загружен: dataset_11 (строк: 1047)
  Загружен: dataset_8 (строк: 1052)
  Загружен: dataset_48 (строк: 1021)
  Загружен: dataset_45 (строк: 1035)
  Загружен: dataset_38 (строк: 1023)
  Загружен: dataset_32 (строк: 1073)
  Загружен: dataset_36 (строк: 1062)
  Загружен: dataset_44 (строк: 1061)
  Загружен: dataset_15 (строк: 1028)

✅ Итоговый размер X_train: (18002, 22)

РАЗМЕРЫ ДАННЫХ ПОСЛЕ ЗАГРУЗКИ
X_train: (18002, 22)
y_train: (8340, 2)
X_test:  (8341, 22)


In [31]:
print("\n" + "="*80)
print("ШАГ 2: АНАЛИЗ СТРУКТУРЫ ДАННЫХ")
print("="*80)

# Проверяем, есть ли ID
possible_ids = ['car_id', 'id', 'vehicle_id', 'auto_id', 'ID']
id_col = None

for pid in possible_ids:
    if pid in X_train.columns and pid in y_train.columns:
        id_col = pid
        break

if id_col is None:
    # Проверяем общие колонки
    common_cols = set(X_train.columns) & set(y_train.columns)
    if common_cols:
        id_col = list(common_cols)[0]
        print(f"🔑 Найден общий ID: '{id_col}'")
    else:
        print("❌ НЕ НАЙДЕН ОБЩИЙ ID!")
        print(f"Колонки X_train: {X_train.columns.tolist()[:10]}")
        print(f"Колонки y_train: {y_train.columns.tolist()}")
        raise ValueError("Нет общего идентификатора!")
else:
    print(f"🔑 Найден ID: '{id_col}'")

# Переименовываем колонку с ценой
if 'Цена' not in y_train.columns:
    price_col = [c for c in y_train.columns if c != id_col][0]
    y_train = y_train.rename(columns={price_col: 'Цена'})
    print(f"💰 Колонка с ценой переименована: '{price_col}' -> 'Цена'")

# Определяем типы колонок
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != id_col]

cat_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()
cat_cols = [c for c in cat_cols if c != id_col]

print(f"\n📊 Числовых признаков: {len(numeric_cols)}")
print(f"📝 Категориальных признаков: {len(cat_cols)}")

if len(numeric_cols) > 0:
    print(f"Числовые: {numeric_cols[:5]}")
if len(cat_cols) > 0:
    print(f"Категориальные: {cat_cols[:5]}")

# Проверяем, сколько раз каждая машина встречается
if id_col in X_train.columns:
    car_counts = X_train[id_col].value_counts()
    print(f"\n📊 Уникальных машин в X_train: {X_train[id_col].nunique()}")
    print(f"   Всего строк: {len(X_train)}")
    print(f"   В среднем {len(X_train)/X_train[id_col].nunique():.2f} записей на машину")
    print(f"   Мин: {car_counts.min()}, Макс: {car_counts.max()}")



ШАГ 2: АНАЛИЗ СТРУКТУРЫ ДАННЫХ
🔑 Найден ID: 'car_id'

📊 Числовых признаков: 4
📝 Категориальных признаков: 17
Числовые: ['Год выпуска', 'Оценка эксперта', 'Количество владельцев', 'Предложение']
Категориальные: ['Бренд', 'Модель', 'Тип машины', 'Полное название', 'Исползование']

📊 Уникальных машин в X_train: 3127
   Всего строк: 18002
   В среднем 5.76 записей на машину
   Мин: 1, Макс: 11


In [32]:
print("\n" + "="*80)
print("ШАГ 3: АГРЕГАЦИЯ ДАННЫХ ПО ID")
print("="*80)

# 3.1. Агрегация числовых признаков
if len(numeric_cols) > 0:
    X_train_agg = X_train.groupby(id_col)[numeric_cols].agg([
        'mean', 'std', 'min', 'max', 'count'
    ]).reset_index()
    
    # Переименовываем колонки
    X_train_agg.columns = [id_col] + [f"{col}_{agg}" for col in numeric_cols for agg in ['mean', 'std', 'min', 'max', 'count']]
    print(f"✅ Числовые признаки агрегированы: {X_train_agg.shape}")
else:
    X_train_agg = pd.DataFrame({id_col: X_train[id_col].unique()})
    print("⚠️ Нет числовых признаков для агрегации")

# 3.2. Добавление категориальных признаков
if len(cat_cols) > 0:
    print("\nДобавление категориальных признаков...")
    cat_features = {}
    for col in cat_cols:
        cat_agg = X_train.groupby(id_col)[col].first().reset_index()
        cat_features[col] = cat_agg
    
    # Объединяем все категории
    cat_df = cat_features[cat_cols[0]]
    for col in cat_cols[1:]:
        cat_df = pd.merge(cat_df, cat_features[col], on=id_col, how='left')
    
    # Добавляем категории к агрегированным данным
    X_train_agg = pd.merge(X_train_agg, cat_df, on=id_col, how='left')
    print(f"✅ Категориальные признаки добавлены: {X_train_agg.shape}")



ШАГ 3: АГРЕГАЦИЯ ДАННЫХ ПО ID
✅ Числовые признаки агрегированы: (3127, 21)

Добавление категориальных признаков...
✅ Категориальные признаки добавлены: (3127, 38)


In [33]:
print("\n" + "="*80)
print("ШАГ 4: ОБЪЕДИНЕНИЕ С ЦЕНАМИ")
print("="*80)

# Проверяем пересечение ID
ids_in_agg = set(X_train_agg[id_col])
ids_in_y = set(y_train[id_col])

print(f"ID в агрегированных данных: {len(ids_in_agg)}")
print(f"ID в y_train: {len(ids_in_y)}")

ids_only_in_y = ids_in_y - ids_in_agg
if ids_only_in_y:
    print(f"⚠️ {len(ids_only_in_y)} машин из y_train отсутствуют в X_train (не будут использованы)")

# Объединяем с ценами
train_full = pd.merge(X_train_agg, y_train, on=id_col, how='inner')
print(f"✅ После объединения с ценами: {train_full.shape}")

# Сохраняем ID
train_ids = train_full[id_col].copy()
X_train_final = train_full.drop(columns=[id_col, 'Цена'])
y_train_final = train_full['Цена']

print(f"\n✅ X_train: {X_train_final.shape}")
print(f"✅ y_train: {y_train_final.shape}")


ШАГ 4: ОБЪЕДИНЕНИЕ С ЦЕНАМИ
ID в агрегированных данных: 3127
ID в y_train: 8340
⚠️ 5213 машин из y_train отсутствуют в X_train (не будут использованы)
✅ После объединения с ценами: (3127, 39)

✅ X_train: (3127, 37)
✅ y_train: (3127,)


In [34]:
print("\n" + "="*80)
print("ШАГ 5: ПОДГОТОВКА ТЕСТОВЫХ ДАННЫХ")
print("="*80)

# Проверяем, есть ли дубликаты в тесте
if X_test[id_col].nunique() < len(X_test):
    print("⚠️ В X_test есть дубликаты. Агрегируем...")
    
    # Агрегируем тест
    if len(numeric_cols) > 0:
        X_test_agg = X_test.groupby(id_col)[numeric_cols].agg([
            'mean', 'std', 'min', 'max', 'count'
        ]).reset_index()
        X_test_agg.columns = [id_col] + [f"{col}_{agg}" for col in numeric_cols for agg in ['mean', 'std', 'min', 'max', 'count']]
    else:
        X_test_agg = pd.DataFrame({id_col: X_test[id_col].unique()})
    
    # Добавляем категории
    if len(cat_cols) > 0:
        cat_test_features = {}
        for col in cat_cols:
            cat_agg = X_test.groupby(id_col)[col].first().reset_index()
            cat_test_features[col] = cat_agg
        
        cat_test_df = cat_test_features[cat_cols[0]]
        for col in cat_cols[1:]:
            cat_test_df = pd.merge(cat_test_df, cat_test_features[col], on=id_col, how='left')
        
        X_test_agg = pd.merge(X_test_agg, cat_test_df, on=id_col, how='left')
else:
    print("✅ В X_test каждая машина уникальна")
    X_test_agg = X_test.copy()

# Сохраняем ID для сабмита
test_ids = X_test_agg[id_col]
X_test_final = X_test_agg.drop(columns=[id_col])

print(f"✅ X_test: {X_test_final.shape}")



ШАГ 5: ПОДГОТОВКА ТЕСТОВЫХ ДАННЫХ
✅ В X_test каждая машина уникальна
✅ X_test: (8341, 21)


In [35]:
print("\n" + "="*80)
print("ШАГ 6: ПРИВЕДЕНИЕ КОЛОНОК")
print("="*80)

train_cols = set(X_train_final.columns)
test_cols = set(X_test_final.columns)

common_cols = train_cols & test_cols
print(f"Общих колонок: {len(common_cols)} из {len(train_cols)}")

# Добавляем недостающие колонки
missing_in_test = train_cols - test_cols
missing_in_train = test_cols - train_cols

if missing_in_test:
    print(f"⚠️ Добавляем в тест: {list(missing_in_test)[:5]}...")
    for col in missing_in_test:
        X_test_final[col] = 0

if missing_in_train:
    print(f"⚠️ Добавляем в train: {list(missing_in_train)[:5]}...")
    for col in missing_in_train:
        X_train_final[col] = 0

# Приводим порядок
X_train_final = X_train_final[sorted(X_test_final.columns)]
X_test_final = X_test_final[sorted(X_test_final.columns)]

print(f"\n✅ Итоговые размеры:")
print(f"   X_train: {X_train_final.shape}")
print(f"   y_train: {y_train_final.shape}")
print(f"   X_test:  {X_test_final.shape}")



ШАГ 6: ПРИВЕДЕНИЕ КОЛОНОК
Общих колонок: 17 из 37
⚠️ Добавляем в тест: ['Предложение_max', 'Год выпуска_mean', 'Предложение_min', 'Количество владельцев_count', 'Количество владельцев_mean']...
⚠️ Добавляем в train: ['Количество владельцев', 'Оценка эксперта', 'Год выпуска', 'Предложение']...

✅ Итоговые размеры:
   X_train: (3127, 41)
   y_train: (3127,)
   X_test:  (8341, 41)


In [36]:
print("\n" + "="*80)
print("ШАГ 7: РАЗДЕЛЕНИЕ НА ОБУЧЕНИЕ И ВАЛИДАЦИЮ")
print("="*80)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_final, y_train_final, 
    test_size=0.2, 
    random_state=42
)

print(f"✅ Train: {X_train_split.shape}")
print(f"✅ Val: {X_val_split.shape}")



ШАГ 7: РАЗДЕЛЕНИЕ НА ОБУЧЕНИЕ И ВАЛИДАЦИЮ
✅ Train: (2501, 41)
✅ Val: (626, 41)


In [37]:
print("\n" + "="*80)
print("ШАГ 8: АНАЛИЗ ДАННЫХ (EDA)")
print("="*80)

# Статистика цен
print("\n📊 Статистика цен:")
print(f"   Средняя: {y_train_final.mean():.2f}")
print(f"   Медиана: {y_train_final.median():.2f}")
print(f"   Мин: {y_train_final.min():.2f}")
print(f"   Макс: {y_train_final.max():.2f}")
print(f"   Стандартное отклонение: {y_train_final.std():.2f}")

# Корреляции числовых признаков
numeric_feature_cols = [c for c in X_train_final.columns if any(agg in c for agg in ['_mean', '_std', '_min', '_max', '_count'])]
if numeric_feature_cols:
    corr_matrix = pd.concat([X_train_final[numeric_feature_cols], y_train_final], axis=1).corr()
    target_corr = corr_matrix['Цена'].sort_values(ascending=False)
    print("\n📈 Топ-5 корреляций с ценой:")
    print(target_corr.head(6))

# Категориальные признаки
cat_cols_final = [c for c in X_train_final.columns if c not in numeric_feature_cols]
print(f"\n📝 Категориальных признаков: {len(cat_cols_final)}")
if cat_cols_final:
    print("Уникальные значения (первые 5):")
    for col in cat_cols_final[:5]:
        print(f"   {col}: {X_train_final[col].nunique()}")



ШАГ 8: АНАЛИЗ ДАННЫХ (EDA)

📊 Статистика цен:
   Средняя: 37612.94
   Медиана: 29488.00
   Мин: 900.00
   Макс: 524580.00
   Стандартное отклонение: 36751.55

📈 Топ-5 корреляций с ценой:
Цена               1.0000
Предложение_min    0.9938
Предложение_mean   0.9938
Предложение_max    0.9938
Год выпуска_max    0.4121
Год выпуска_min    0.4121
Name: Цена, dtype: float64

📝 Категориальных признаков: 21
Уникальные значения (первые 5):
   Бренд: 59
   Год выпуска: 1
   Двери: 8
   Двигатель: 77
   Исползование: 3


In [38]:
print("\n" + "="*80)
print("ШАГ 8.5: ПРЕДОБРАБОТКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ")
print("="*80)

# Находим все строковые колонки
cat_features = X_train_split.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Найдено категориальных колонок: {len(cat_features)}")
if cat_features:
    print(f"Примеры: {cat_features[:5]}")

# Для CatBoost не нужно кодировать - он сам умеет
# Для остальных моделей кодируем
X_train_encoded = X_train_split.copy()
X_val_encoded = X_val_split.copy()
X_test_encoded = X_test_final.copy()

label_encoders = {}
for col in cat_features:
    print(f"  Кодируем: {col}")
    le = LabelEncoder()
    
    # Обучаем на всех данных (train + val + test)
    all_values = pd.concat([
        X_train_split[col], 
        X_val_split[col], 
        X_test_final[col]
    ], axis=0).astype(str)
    le.fit(all_values)
    
    # Преобразуем
    X_train_encoded[col] = le.transform(X_train_split[col].astype(str))
    X_val_encoded[col] = le.transform(X_val_split[col].astype(str))
    X_test_encoded[col] = le.transform(X_test_final[col].astype(str))
    
    label_encoders[col] = le

print(f"✅ Все категориальные признаки закодированы!")


ШАГ 8.5: ПРЕДОБРАБОТКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ
Найдено категориальных колонок: 17
Примеры: ['Бренд', 'Двери', 'Двигатель', 'Исползование', 'КПП']
  Кодируем: Бренд
  Кодируем: Двери
  Кодируем: Двигатель
  Кодируем: Исползование
  Кодируем: КПП
  Кодируем: Количество кресел
  Кодируем: Количество цилиндров
  Кодируем: Локация
  Кодируем: Модель
  Кодируем: Полное название
  Кодируем: Привод
  Кодируем: Пробег
  Кодируем: Расход
  Кодируем: Тип кузова
  Кодируем: Тип машины
  Кодируем: Топливо
  Кодируем: Цвет
✅ Все категориальные признаки закодированы!


In [39]:
print("\n" + "="*80)
print("ШАГ 9: ОБУЧЕНИЕ МОДЕЛЕЙ")
print("="*80)

results = {}

# 9.1. CATBOOST (работает со строками!)
print("\n🐱 CatBoost (нативная поддержка строк)...")
try:
    cat_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False,
        cat_features=cat_features  # указываем категориальные колонки
    )
    cat_model.fit(X_train_split, y_train_split)
    cat_pred = cat_model.predict(X_val_split)
    cat_mae = mean_absolute_error(y_val_split, cat_pred)
    cat_rmse = np.sqrt(mean_squared_error(y_val_split, cat_pred))
    cat_r2 = r2_score(y_val_split, cat_pred)
    results['CatBoost'] = {'MAE': cat_mae, 'RMSE': cat_rmse, 'R2': cat_r2}
    print(f"   ✅ MAE: {cat_mae:.2f}, RMSE: {cat_rmse:.2f}, R2: {cat_r2:.4f}")
except Exception as e:
    print(f"   ❌ Ошибка: {e}")



ШАГ 9: ОБУЧЕНИЕ МОДЕЛЕЙ

🐱 CatBoost (нативная поддержка строк)...
   ❌ Ошибка: Invalid type for cat_feature[non-default value idx=2,feature_idx=7]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.


In [40]:
print("\n⚡ XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_encoded, y_train_split)
xgb_pred = xgb_model.predict(X_val_encoded)
xgb_mae = mean_absolute_error(y_val_split, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_val_split, xgb_pred))
xgb_r2 = r2_score(y_val_split, xgb_pred)
results['XGBoost'] = {'MAE': xgb_mae, 'RMSE': xgb_rmse, 'R2': xgb_r2}
print(f"   ✅ MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}, R2: {xgb_r2:.4f}")



⚡ XGBoost...


   ✅ MAE: 3180.87, RMSE: 8540.53, R2: 0.9496


In [41]:
print("\n🌲 Random Forest...")
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_encoded, y_train_split)
rf_pred = rf.predict(X_val_encoded)
rf_mae = mean_absolute_error(y_val_split, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val_split, rf_pred))
rf_r2 = r2_score(y_val_split, rf_pred)
results['Random Forest'] = {'MAE': rf_mae, 'RMSE': rf_rmse, 'R2': rf_r2}
print(f"   ✅ MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R2: {rf_r2:.4f}")



🌲 Random Forest...
   ✅ MAE: 2818.80, RMSE: 5732.21, R2: 0.9773


In [42]:
print("\n" + "="*80)
print("ШАГ 10: ВЫБОР ЛУЧШЕЙ МОДЕЛИ")
print("="*80)

results_df = pd.DataFrame(results).T
print("\nРезультаты всех моделей:")
print(results_df.sort_values('MAE'))

best_model_name = results_df.sort_values('MAE').index[0]
print(f"\n🏆 ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
print(f"   MAE: {results_df.loc[best_model_name, 'MAE']:.2f}")
print(f"   R2:  {results_df.loc[best_model_name, 'R2']:.4f}")

# Сохраняем лучшую модель и соответствующие данные
if best_model_name == 'CatBoost':
    best_model = cat_model
    X_test_for_pred = X_test_final  # CatBoost использует строки
elif best_model_name == 'LightGBM':
    best_model = lgb_model
    X_test_for_pred = X_test_encoded
elif best_model_name == 'XGBoost':
    best_model = xgb_model
    X_test_for_pred = X_test_encoded
else:  # Random Forest
    best_model = rf
    X_test_for_pred = X_test_encoded



ШАГ 10: ВЫБОР ЛУЧШЕЙ МОДЕЛИ

Результаты всех моделей:
                    MAE      RMSE     R2
Random Forest 2818.7997 5732.2079 0.9773
XGBoost       3180.8708 8540.5302 0.9496

🏆 ЛУЧШАЯ МОДЕЛЬ: Random Forest
   MAE: 2818.80
   R2:  0.9773


In [44]:
print("\n" + "="*80)
print("ШАГ 11: ПРЕДСКАЗАНИЕ НА ТЕСТОВЫХ ДАННЫХ")
print("="*80)

# Проверяем, что X_test_for_pred не содержит строк
print(f"\nПроверка типов данных в X_test_for_pred:")
str_cols_test = X_test_for_pred.select_dtypes(include=['object', 'str']).columns.tolist()
if str_cols_test:
    print(f"⚠️ Найдены строковые колонки: {str_cols_test}")
    print("   Преобразуем их в числа через LabelEncoder...")
    
    # Кодируем строковые колонки в тесте
    for col in str_cols_test:
        if col in label_encoders:
            # Используем уже обученный encoder
            X_test_for_pred[col] = label_encoders[col].transform(X_test_for_pred[col].astype(str))
        else:
            # Если encoder не найден, создаем новый
            le = LabelEncoder()
            # Обучаем на всех данных
            all_vals = pd.concat([
                X_train_split[col], 
                X_val_split[col], 
                X_test_for_pred[col]
            ], axis=0).astype(str)
            le.fit(all_vals)
            X_test_for_pred[col] = le.transform(X_test_for_pred[col].astype(str))
            label_encoders[col] = le
    
    print("✅ Все строковые колонки закодированы!")

# Дополнительная проверка: заполняем NaN в тесте
if X_test_for_pred.isnull().any().any():
    print("\n⚠️ Найдены пропуски в тесте. Заполняем...")
    X_test_for_pred = X_test_for_pred.fillna(0)

# Проверяем, что все данные числовые
print(f"\nТипы данных после обработки:")
print(X_test_for_pred.dtypes.value_counts())

# Проверяем, что нет строковых колонок
str_cols_check = X_test_for_pred.select_dtypes(include=['object', 'str']).columns.tolist()
if str_cols_check:
    print(f"❌ ОСТАЛИСЬ СТРОКОВЫЕ КОЛОНКИ: {str_cols_check}")
    print("Пробуем принудительно преобразовать...")
    for col in str_cols_check:
        X_test_for_pred[col] = pd.to_numeric(X_test_for_pred[col], errors='coerce').fillna(0)
    print("✅ Принудительно преобразованы в числа")

# Делаем предсказание
print("\nДелаем предсказание...")
test_predictions = best_model.predict(X_test_for_pred)

print(f"\n✅ Предсказано {len(test_predictions)} машин")
print(f"   Диапазон: от {test_predictions.min():.2f} до {test_predictions.max():.2f}")
print(f"   Среднее: {test_predictions.mean():.2f}")


ШАГ 11: ПРЕДСКАЗАНИЕ НА ТЕСТОВЫХ ДАННЫХ

Проверка типов данных в X_test_for_pred:
⚠️ Найдены строковые колонки: ['Предложение']
   Преобразуем их в числа через LabelEncoder...
✅ Все строковые колонки закодированы!

Типы данных после обработки:
int64      40
float64     1
Name: count, dtype: int64

Делаем предсказание...

✅ Предсказано 8341 машин
   Диапазон: от 2997.37 до 3270.12
   Среднее: 3142.23


In [46]:
print("\n" + "="*80)
print("ШАГ 12: СОЗДАНИЕ САБМИТА")
print("="*80)

submission = pd.DataFrame({
    id_col: test_ids,
    'Цена': test_predictions
})

print(f"✅ Сабмит создан: {submission.shape}")

# Сохраняем
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
filename = "submission.csv"
submission.to_csv(filename, index=False)
print(f"\n💾 Сабмит сохранен: {filename}")



ШАГ 12: СОЗДАНИЕ САБМИТА
✅ Сабмит создан: (8341, 2)

💾 Сабмит сохранен: submission.csv


In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

import os
from datetime import datetime
import glob
import zipfile

# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("="*80)
print("🚗 ПРЕДСКАЗАНИЕ ЦЕН АВТОМОБИЛЕЙ - ПОЛНЫЙ ПАЙПЛАЙН")
print("="*80)

# ==========================================
# ШАГ 1: ЗАГРУЗКА ДАННЫХ
# ==========================================
print("\n" + "="*80)
print("ШАГ 1: ЗАГРУЗКА ДАННЫХ")
print("="*80)

def load_data_from_folders_glob(base_path, filename="data.csv"):
    """Загружает все data.csv из подпапок base_path"""
    csv_files = glob.glob(os.path.join(base_path, "**", filename), recursive=True)
    print(f"Найдено файлов: {len(csv_files)}")
    
    dataframes = []
    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)
            dataframes.append(df)
            print(f"  Загружен: {os.path.basename(os.path.dirname(file_path))} (строк: {len(df)})")
        except Exception as e:
            print(f"  Ошибка при загрузке {file_path}: {e}")
    
    if dataframes:
        X_train = pd.concat(dataframes, ignore_index=True)
        print(f"\n✅ Итоговый размер X_train: {X_train.shape}")
        return X_train
    else:
        print("❌ Файлы data.csv не найдены!")
        return pd.DataFrame()

# Загружаем данные
base_directory = "./"
X_train = load_data_from_folders_glob(base_directory)
y_train = pd.read_csv('./y_train_base.csv')
X_test = pd.read_csv('./X_test_base.csv')

print("\n" + "="*80)
print("РАЗМЕРЫ ДАННЫХ ПОСЛЕ ЗАГРУЗКИ")
print("="*80)
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")

# ==========================================
# ШАГ 2: АНАЛИЗ СТРУКТУРЫ ДАННЫХ
# ==========================================
print("\n" + "="*80)
print("ШАГ 2: АНАЛИЗ СТРУКТУРЫ ДАННЫХ")
print("="*80)

# Проверяем, есть ли ID
possible_ids = ['car_id', 'id', 'vehicle_id', 'auto_id', 'ID']
id_col = None

for pid in possible_ids:
    if pid in X_train.columns and pid in y_train.columns:
        id_col = pid
        break

if id_col is None:
    common_cols = set(X_train.columns) & set(y_train.columns)
    if common_cols:
        id_col = list(common_cols)[0]
        print(f"🔑 Найден общий ID: '{id_col}'")
    else:
        print("❌ НЕ НАЙДЕН ОБЩИЙ ID!")
        print(f"Колонки X_train: {X_train.columns.tolist()[:10]}")
        print(f"Колонки y_train: {y_train.columns.tolist()}")
        raise ValueError("Нет общего идентификатора!")
else:
    print(f"🔑 Найден ID: '{id_col}'")

# Переименовываем колонку с ценой
if 'Цена' not in y_train.columns:
    price_col = [c for c in y_train.columns if c != id_col][0]
    y_train = y_train.rename(columns={price_col: 'Цена'})
    print(f"💰 Колонка с ценой переименована: '{price_col}' -> 'Цена'")

# Определяем типы колонок
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != id_col]

cat_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()
cat_cols = [c for c in cat_cols if c != id_col]

print(f"\n📊 Числовых признаков: {len(numeric_cols)}")
print(f"📝 Категориальных признаков: {len(cat_cols)}")

if len(numeric_cols) > 0:
    print(f"Числовые: {numeric_cols[:5]}")
if len(cat_cols) > 0:
    print(f"Категориальные: {cat_cols[:5]}")

# Проверяем, сколько раз каждая машина встречается
if id_col in X_train.columns:
    car_counts = X_train[id_col].value_counts()
    print(f"\n📊 Уникальных машин в X_train: {X_train[id_col].nunique()}")
    print(f"   Всего строк: {len(X_train)}")
    print(f"   В среднем {len(X_train)/X_train[id_col].nunique():.2f} записей на машину")
    print(f"   Мин: {car_counts.min()}, Макс: {car_counts.max()}")

# ==========================================
# ШАГ 3: АГРЕГАЦИЯ ДАННЫХ ПО ID
# ==========================================
print("\n" + "="*80)
print("ШАГ 3: АГРЕГАЦИЯ ДАННЫХ ПО ID")
print("="*80)

# 3.1. Агрегация числовых признаков
if len(numeric_cols) > 0:
    X_train_agg = X_train.groupby(id_col)[numeric_cols].agg([
        'mean', 'std', 'min', 'max', 'count'
    ]).reset_index()
    
    X_train_agg.columns = [id_col] + [f"{col}_{agg}" for col in numeric_cols for agg in ['mean', 'std', 'min', 'max', 'count']]
    print(f"✅ Числовые признаки агрегированы: {X_train_agg.shape}")
else:
    X_train_agg = pd.DataFrame({id_col: X_train[id_col].unique()})
    print("⚠️ Нет числовых признаков для агрегации")

# 3.2. Добавление категориальных признаков
if len(cat_cols) > 0:
    print("\nДобавление категориальных признаков...")
    cat_features = {}
    for col in cat_cols:
        cat_agg = X_train.groupby(id_col)[col].first().reset_index()
        cat_features[col] = cat_agg
    
    cat_df = cat_features[cat_cols[0]]
    for col in cat_cols[1:]:
        cat_df = pd.merge(cat_df, cat_features[col], on=id_col, how='left')
    
    X_train_agg = pd.merge(X_train_agg, cat_df, on=id_col, how='left')
    print(f"✅ Категориальные признаки добавлены: {X_train_agg.shape}")

# ==========================================
# ШАГ 4: ОБЪЕДИНЕНИЕ С ЦЕНАМИ
# ==========================================
print("\n" + "="*80)
print("ШАГ 4: ОБЪЕДИНЕНИЕ С ЦЕНАМИ")
print("="*80)

ids_in_agg = set(X_train_agg[id_col])
ids_in_y = set(y_train[id_col])

print(f"ID в агрегированных данных: {len(ids_in_agg)}")
print(f"ID в y_train: {len(ids_in_y)}")

ids_only_in_y = ids_in_y - ids_in_agg
if ids_only_in_y:
    print(f"⚠️ {len(ids_only_in_y)} машин из y_train отсутствуют в X_train (не будут использованы)")

# Объединяем с ценами
train_full = pd.merge(X_train_agg, y_train, on=id_col, how='inner')
print(f"✅ После объединения с ценами: {train_full.shape}")

# Сохраняем ID
train_ids = train_full[id_col].copy()
X_train_final = train_full.drop(columns=[id_col, 'Цена'])
y_train_final = train_full['Цена']

print(f"\n✅ X_train: {X_train_final.shape}")
print(f"✅ y_train: {y_train_final.shape}")

# ==========================================
# ШАГ 5: ПОДГОТОВКА ТЕСТОВЫХ ДАННЫХ
# ==========================================
print("\n" + "="*80)
print("ШАГ 5: ПОДГОТОВКА ТЕСТОВЫХ ДАННЫХ")
print("="*80)

# Проверяем, есть ли дубликаты в тесте
if X_test[id_col].nunique() < len(X_test):
    print("⚠️ В X_test есть дубликаты. Агрегируем...")
    
    if len(numeric_cols) > 0:
        X_test_agg = X_test.groupby(id_col)[numeric_cols].agg([
            'mean', 'std', 'min', 'max', 'count'
        ]).reset_index()
        X_test_agg.columns = [id_col] + [f"{col}_{agg}" for col in numeric_cols for agg in ['mean', 'std', 'min', 'max', 'count']]
    else:
        X_test_agg = pd.DataFrame({id_col: X_test[id_col].unique()})
    
    if len(cat_cols) > 0:
        cat_test_features = {}
        for col in cat_cols:
            cat_agg = X_test.groupby(id_col)[col].first().reset_index()
            cat_test_features[col] = cat_agg
        
        cat_test_df = cat_test_features[cat_cols[0]]
        for col in cat_cols[1:]:
            cat_test_df = pd.merge(cat_test_df, cat_test_features[col], on=id_col, how='left')
        
        X_test_agg = pd.merge(X_test_agg, cat_test_df, on=id_col, how='left')
else:
    print("✅ В X_test каждая машина уникальна")
    X_test_agg = X_test.copy()

# Сохраняем ID для сабмита
test_ids = X_test_agg[id_col]
X_test_final = X_test_agg.drop(columns=[id_col])

print(f"✅ X_test: {X_test_final.shape}")

# ==========================================
# ШАГ 6: ПРИВЕДЕНИЕ КОЛОНОК
# ==========================================
print("\n" + "="*80)
print("ШАГ 6: ПРИВЕДЕНИЕ КОЛОНОК")
print("="*80)

train_cols = set(X_train_final.columns)
test_cols = set(X_test_final.columns)

common_cols = train_cols & test_cols
print(f"Общих колонок: {len(common_cols)} из {len(train_cols)}")

# Добавляем недостающие колонки
missing_in_test = train_cols - test_cols
missing_in_train = test_cols - train_cols

if missing_in_test:
    print(f"⚠️ Добавляем в тест: {list(missing_in_test)[:5]}...")
    for col in missing_in_test:
        X_test_final[col] = 0

if missing_in_train:
    print(f"⚠️ Добавляем в train: {list(missing_in_train)[:5]}...")
    for col in missing_in_train:
        X_train_final[col] = 0

# Приводим порядок
X_train_final = X_train_final[sorted(X_test_final.columns)]
X_test_final = X_test_final[sorted(X_test_final.columns)]

print(f"\n✅ Итоговые размеры:")
print(f"   X_train: {X_train_final.shape}")
print(f"   y_train: {y_train_final.shape}")
print(f"   X_test:  {X_test_final.shape}")

# ==========================================
# ШАГ 7: РАЗДЕЛЕНИЕ НА ОБУЧЕНИЕ И ВАЛИДАЦИЮ
# ==========================================
print("\n" + "="*80)
print("ШАГ 7: РАЗДЕЛЕНИЕ НА ОБУЧЕНИЕ И ВАЛИДАЦИЮ")
print("="*80)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_final, y_train_final, 
    test_size=0.2, 
    random_state=42
)

print(f"✅ Train: {X_train_split.shape}")
print(f"✅ Val: {X_val_split.shape}")

# ==========================================
# ШАГ 8: АНАЛИЗ ДАННЫХ (EDA)
# ==========================================
print("\n" + "="*80)
print("ШАГ 8: АНАЛИЗ ДАННЫХ (EDA)")
print("="*80)

print("\n📊 Статистика цен:")
print(f"   Средняя: {y_train_final.mean():.2f}")
print(f"   Медиана: {y_train_final.median():.2f}")
print(f"   Мин: {y_train_final.min():.2f}")
print(f"   Макс: {y_train_final.max():.2f}")
print(f"   Стандартное отклонение: {y_train_final.std():.2f}")

# Корреляции числовых признаков
numeric_feature_cols = [c for c in X_train_final.columns if any(agg in c for agg in ['_mean', '_std', '_min', '_max', '_count'])]
if numeric_feature_cols:
    corr_matrix = pd.concat([X_train_final[numeric_feature_cols], y_train_final], axis=1).corr()
    target_corr = corr_matrix['Цена'].sort_values(ascending=False)
    print("\n📈 Топ-5 корреляций с ценой:")
    print(target_corr.head(6))

# Категориальные признаки
cat_cols_final = [c for c in X_train_final.columns if c not in numeric_feature_cols]
print(f"\n📝 Категориальных признаков: {len(cat_cols_final)}")
if cat_cols_final:
    print("Уникальные значения (первые 5):")
    for col in cat_cols_final[:5]:
        print(f"   {col}: {X_train_final[col].nunique()}")

# ==========================================
# ШАГ 8.5: ПРЕДОБРАБОТКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ
# ==========================================
print("\n" + "="*80)
print("ШАГ 8.5: ПРЕДОБРАБОТКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ")
print("="*80)

cat_features = X_train_split.select_dtypes(include=['object', 'str']).columns.tolist()
print(f"Найдено категориальных колонок: {len(cat_features)}")
if cat_features:
    print(f"Примеры: {cat_features[:5]}")

X_train_encoded = X_train_split.copy()
X_val_encoded = X_val_split.copy()
X_test_encoded = X_test_final.copy()

label_encoders = {}
for col in cat_features:
    print(f"  Кодируем: {col}")
    le = LabelEncoder()
    
    all_values = pd.concat([
        X_train_split[col], 
        X_val_split[col], 
        X_test_final[col]
    ], axis=0).astype(str)
    le.fit(all_values)
    
    X_train_encoded[col] = le.transform(X_train_split[col].astype(str))
    X_val_encoded[col] = le.transform(X_val_split[col].astype(str))
    X_test_encoded[col] = le.transform(X_test_final[col].astype(str))
    
    label_encoders[col] = le

print(f"✅ Все категориальные признаки закодированы!")

# ==========================================
# ШАГ 9: ОБУЧЕНИЕ МОДЕЛЕЙ
# ==========================================
print("\n" + "="*80)
print("ШАГ 9: ОБУЧЕНИЕ МОДЕЛЕЙ")
print("="*80)

results = {}

# 9.1. CATBOOST
print("\n🐱 CatBoost...")
try:
    cat_model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False,
        cat_features=cat_features
    )
    cat_model.fit(X_train_split, y_train_split)
    cat_pred = cat_model.predict(X_val_split)
    cat_mae = mean_absolute_error(y_val_split, cat_pred)
    cat_rmse = np.sqrt(mean_squared_error(y_val_split, cat_pred))
    cat_r2 = r2_score(y_val_split, cat_pred)
    cat_mape = mean_absolute_percentage_error(y_val_split, cat_pred)
    results['CatBoost'] = {'MAE': cat_mae, 'RMSE': cat_rmse, 'R2': cat_r2, 'MAPE': cat_mape}
    print(f"   ✅ MAE: {cat_mae:.2f}, RMSE: {cat_rmse:.2f}, R2: {cat_r2:.4f}, MAPE: {cat_mape:.4f}")
except Exception as e:
    print(f"   ❌ Ошибка: {e}")

# 9.2. LIGHTGBM
print("\n💡 LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_model.fit(X_train_encoded, y_train_split)
lgb_pred = lgb_model.predict(X_val_encoded)
lgb_mae = mean_absolute_error(y_val_split, lgb_pred)
lgb_rmse = np.sqrt(mean_squared_error(y_val_split, lgb_pred))
lgb_r2 = r2_score(y_val_split, lgb_pred)
lgb_mape = mean_absolute_percentage_error(y_val_split, lgb_pred)
results['LightGBM'] = {'MAE': lgb_mae, 'RMSE': lgb_rmse, 'R2': lgb_r2, 'MAPE': lgb_mape}
print(f"   ✅ MAE: {lgb_mae:.2f}, RMSE: {lgb_rmse:.2f}, R2: {lgb_r2:.4f}, MAPE: {lgb_mape:.4f}")

# 9.3. XGBOOST
print("\n⚡ XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_encoded, y_train_split)
xgb_pred = xgb_model.predict(X_val_encoded)
xgb_mae = mean_absolute_error(y_val_split, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_val_split, xgb_pred))
xgb_r2 = r2_score(y_val_split, xgb_pred)
xgb_mape = mean_absolute_percentage_error(y_val_split, xgb_pred)
results['XGBoost'] = {'MAE': xgb_mae, 'RMSE': xgb_rmse, 'R2': xgb_r2, 'MAPE': xgb_mape}
print(f"   ✅ MAE: {xgb_mae:.2f}, RMSE: {xgb_rmse:.2f}, R2: {xgb_r2:.4f}, MAPE: {xgb_mape:.4f}")

# 9.4. RANDOM FOREST
print("\n🌲 Random Forest...")
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_encoded, y_train_split)
rf_pred = rf.predict(X_val_encoded)
rf_mae = mean_absolute_error(y_val_split, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val_split, rf_pred))
rf_r2 = r2_score(y_val_split, rf_pred)
rf_mape = mean_absolute_percentage_error(y_val_split, rf_pred)
results['Random Forest'] = {'MAE': rf_mae, 'RMSE': rf_rmse, 'R2': rf_r2, 'MAPE': rf_mape}
print(f"   ✅ MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R2: {rf_r2:.4f}, MAPE: {rf_mape:.4f}")

# ==========================================
# ШАГ 10: ВЫБОР ЛУЧШЕЙ МОДЕЛИ (по MAPE)
# ==========================================
print("\n" + "="*80)
print("ШАГ 10: ВЫБОР ЛУЧШЕЙ МОДЕЛИ")
print("="*80)

results_df = pd.DataFrame(results).T
print("\nРезультаты всех моделей:")
print(results_df.sort_values('MAPE'))

best_model_name = results_df.sort_values('MAPE').index[0]
print(f"\n🏆 ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
print(f"   MAE: {results_df.loc[best_model_name, 'MAE']:.2f}")
print(f"   MAPE: {results_df.loc[best_model_name, 'MAPE']:.4f}")

# Сохраняем лучшую модель
if best_model_name == 'CatBoost':
    best_model = cat_model
    X_test_for_pred = X_test_final
elif best_model_name == 'LightGBM':
    best_model = lgb_model
    X_test_for_pred = X_test_encoded
elif best_model_name == 'XGBoost':
    best_model = xgb_model
    X_test_for_pred = X_test_encoded
else:
    best_model = rf
    X_test_for_pred = X_test_encoded

# ==========================================
# ШАГ 11: ПРЕДСКАЗАНИЕ НА ТЕСТЕ
# ==========================================
print("\n" + "="*80)
print("ШАГ 11: ПРЕДСКАЗАНИЕ НА ТЕСТОВЫХ ДАННЫХ")
print("="*80)

# Проверяем и кодируем строки в тесте
str_cols_test = X_test_for_pred.select_dtypes(include=['object', 'str']).columns.tolist()
if str_cols_test:
    print(f"⚠️ Найдены строковые колонки: {str_cols_test[:5]}")
    for col in str_cols_test:
        if col in label_encoders:
            X_test_for_pred[col] = label_encoders[col].transform(X_test_for_pred[col].astype(str))
        else:
            le = LabelEncoder()
            all_vals = pd.concat([X_train_split[col], X_val_split[col], X_test_for_pred[col]], axis=0).astype(str)
            le.fit(all_vals)
            X_test_for_pred[col] = le.transform(X_test_for_pred[col].astype(str))
            label_encoders[col] = le

# Заполняем пропуски
X_test_for_pred = X_test_for_pred.fillna(0)

# Принудительно конвертируем все в числа
for col in X_test_for_pred.columns:
    if X_test_for_pred[col].dtype == 'object':
        X_test_for_pred[col] = pd.to_numeric(X_test_for_pred[col], errors='coerce').fillna(0)

# Делаем предсказание
test_predictions = best_model.predict(X_test_for_pred)

print(f"\n✅ Предсказано {len(test_predictions)} машин")
print(f"   Диапазон: от {test_predictions.min():.2f} до {test_predictions.max():.2f}")
print(f"   Среднее: {test_predictions.mean():.2f}")

# ==========================================
# ШАГ 12: СОЗДАНИЕ САБМИТА
# ==========================================
print("\n" + "="*80)
print("ШАГ 12: СОЗДАНИЕ САБМИТА")
print("="*80)

# Проверяем предсказания
print(f"Проверка предсказаний:")
print(f"  shape: {test_predictions.shape}")
print(f"  dtype: {test_predictions.dtype}")
print(f"  min: {test_predictions.min():.2f}")
print(f"  max: {test_predictions.max():.2f}")
print(f"  mean: {test_predictions.mean():.2f}")
print(f"  NaN count: {np.isnan(test_predictions).sum()}")

# Заменяем NaN и отрицательные значения
test_predictions_clean = test_predictions.copy()
if np.isnan(test_predictions_clean).any():
    print("⚠️ Найдены NaN. Заменяем на медиану...")
    test_predictions_clean = np.nan_to_num(test_predictions_clean, nan=y_train_final.median())

if (test_predictions_clean < 0).any():
    print("⚠️ Найдены отрицательные предсказания. Заменяем на 0...")
    test_predictions_clean = np.maximum(test_predictions_clean, 0)

# Приводим к int (если цена должна быть целой)
# test_predictions_clean = np.round(test_predictions_clean).astype(int)

# Создаем сабмит
submission = pd.DataFrame({
    'id': test_ids.values,
    'price': test_predictions_clean
})

print(f"\n✅ Сабмит создан: {submission.shape}")
print(f"   id: от {submission['id'].min()} до {submission['id'].max()}")
print(f"   price: от {submission['price'].min():.2f} до {submission['price'].max():.2f}")
print(f"   price mean: {submission['price'].mean():.2f}")

# Сохраняем submission.csv
submission.to_csv('submission.csv', index=False)
print(f"\n💾 Сабмит сохранен: submission.csv")

# Проверяем файл
print(f"\n📋 Проверка файла submission.csv:")
df_check = pd.read_csv('submission.csv')
print(f"   shape: {df_check.shape}")
print(f"   columns: {df_check.columns.tolist()}")
print(f"   dtypes: {df_check.dtypes}")
print(f"   head: \n{df_check.head(10)}")

# ==========================================
# ШАГ 13: СОЗДАНИЕ submission.zip
# ==========================================
print("\n" + "="*80)
print("ШАГ 13: СОЗДАНИЕ submission.zip")
print("="*80)

# Удаляем старый архив, если существует
if os.path.exists('submission.zip'):
    os.remove('submission.zip')

# Создаем новый архив
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('submission.csv')

# Проверяем архив
print(f"✅ submission.zip создан!")
print(f"   Размер: {os.path.getsize('submission.zip')} байт")

# Проверяем содержимое архива
with zipfile.ZipFile('submission.zip', 'r') as zipf:
    print(f"   Содержимое: {zipf.namelist()}")

# ==========================================
# ШАГ 14: ФИНАЛЬНАЯ СТАТИСТИКА
# ==========================================
print("\n" + "="*80)
print("🎯 ФИНАЛЬНАЯ СТАТИСТИКА")
print("="*80)

print(f"""
📊 Итоги:
   ✅ Машин для обучения: {len(y_train_final)}
   ✅ Машин для предсказания: {len(X_test_final)}
   ✅ Признаков: {X_train_final.shape[1]}
   ✅ Категориальных признаков: {len(cat_features)}
   ✅ Лучшая модель: {best_model_name}
   ✅ MAPE на валидации: {results_df.loc[best_model_name, 'MAPE']:.4f}
   ✅ MAE на валидации: {results_df.loc[best_model_name, 'MAE']:.2f}
   ✅ Файл сабмита: submission.csv
   ✅ Архив: submission.zip
""")

print("="*80)
print("✅ ПАЙПЛАЙН ЗАВЕРШЕН!")
print("📤 Загрузи submission.zip на платформу!")
print("="*80)

# Дополнительно: создаем отдельный файл с метаданными для отладки
with open('submission_info.txt', 'w') as f:
    f.write(f"Model: {best_model_name}\n")
    f.write(f"MAPE: {results_df.loc[best_model_name, 'MAPE']:.4f}\n")
    f.write(f"MAE: {results_df.loc[best_model_name, 'MAE']:.2f}\n")
    f.write(f"Train size: {len(y_train_final)}\n")
    f.write(f"Test size: {len(X_test_final)}\n")
    f.write(f"Features: {X_train_final.shape[1]}\n")
    f.write(f"Predictions min: {test_predictions_clean.min():.2f}\n")
    f.write(f"Predictions max: {test_predictions_clean.max():.2f}\n")
    f.write(f"Predictions mean: {test_predictions_clean.mean():.2f}\n")
    print("✅ submission_info.txt создан")

🚗 ПРЕДСКАЗАНИЕ ЦЕН АВТОМОБИЛЕЙ - ПОЛНЫЙ ПАЙПЛАЙН

ШАГ 1: ЗАГРУЗКА ДАННЫХ
Найдено файлов: 36
  Загружен: dataset_10 (строк: 1069)
  Загружен: dataset_2 (строк: 1026)
  Загружен: dataset_7 (строк: 1079)
  Загружен: dataset_10 (строк: 1069)
  Загружен: dataset_2 (строк: 1026)
  Загружен: dataset_7 (строк: 1079)
  Загружен: dataset_30 (строк: 1108)
  Загружен: dataset_1 (строк: 174)
  Загружен: dataset_40 (строк: 1071)
  Загружен: dataset_29 (строк: 1063)
  Загружен: dataset_21 (строк: 996)
  Загружен: dataset_12 (строк: 1014)
  Загружен: dataset_11 (строк: 1047)
  Загружен: dataset_8 (строк: 1052)
  Загружен: dataset_48 (строк: 1021)
  Загружен: dataset_45 (строк: 1035)
  Загружен: dataset_38 (строк: 1023)
  Загружен: dataset_32 (строк: 1073)
  Загружен: dataset_36 (строк: 1062)
  Загружен: dataset_44 (строк: 1061)
  Загружен: dataset_15 (строк: 1028)
  Загружен: dataset_30 (строк: 1108)
  Загружен: dataset_1 (строк: 174)
  Загружен: dataset_40 (строк: 1071)
  Загружен: dataset_29 (строк: